In [2]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Cargar el modelo previamente entrenado
model = load_model("model_kadle.h5")

# Mapear las emociones a etiquetas comprensibles
emotion_mapper = {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'happiness', 4: 'sadness', 5: 'surprise', 6: 'neutral'}


/Users/sofiaguerrero/miniconda3/lib/python3.12/site-packages/keras/src/optimizers/base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [ ]:
# Inicializar la cámara
cap = cv2.VideoCapture(0)

# Verificar si la cámara se abrió correctamente
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara.")
    exit()

# Configuración de la ventana para mostrar el video
cv2.namedWindow("Reconocimiento de Emociones en Tiempo Real")

# Función para preprocesar las imágenes antes de pasarlas al modelo
def preprocess_image(img):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convertir a escala de grises
    resized_img = cv2.resize(gray_img, (48, 48))  # Redimensionar a 48x48 píxeles
    rgb_img = cv2.cvtColor(resized_img, cv2.COLOR_GRAY2RGB)  # Convertir de nuevo a RGB
    normalized_img = rgb_img / 255.0  # Normalizar
    return np.expand_dims(normalized_img, axis=0)  # Expandir dimensiones para que coincida con el input del modelo



In [ ]:
while True:
    ret, frame = cap.read()  # Capturar frame de la cámara
    if not ret:
        print("Error: No se pudo capturar el frame.")
        break

    # Preprocesar la imagen para el modelo
    preprocessed_frame = preprocess_image(frame)

    # Predecir la emoción
    prediction = model.predict(preprocessed_frame)
    predicted_emotion = emotion_mapper[np.argmax(prediction)]

    # Mostrar la emoción en el video en tiempo real
    cv2.putText(frame, f'Emocion: {predicted_emotion}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Mostrar el video con la predicción
    cv2.imshow("Reconocimiento de Emociones en Tiempo Real", frame)

    # Presionar 'q' para salir del loop
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar la cámara y cerrar ventanas
# Configurar la cámara para una resolución más baja (opcional para mejorar la velocidad)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

cap.release()
cv2.destroyAllWindows()


In [3]:
import cv2
import numpy as np
import os
from tensorflow.keras.models import load_model

# Cargar el modelo previamente entrenado
model = load_model("model_kadle.h5")

# Mapear las emociones a etiquetas comprensibles
emotion_mapper = {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'happiness', 4: 'sadness', 5: 'surprise', 6: 'neutral'}

# Cargar el clasificador de Haar para detección de rostros
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Crear carpeta para guardar las imágenes preprocesadas, si no existe
output_folder = 'preprocessed_images'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Inicializar la cámara
cap = cv2.VideoCapture(0)

# Contador de imágenes guardadas
image_counter = 0

# Función para detectar la cara y preprocesar la imagen
def preprocess_image(img):
    # Detectar caras en el frame
    faces = face_cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) > 0:
        # Tomar la primera cara detectada
        x, y, w, h = faces[0]
        face_img = img[y:y+h, x:x+w]  # Recortar la región de la cara
        
        # Convertir la imagen a escala de grises
        gray_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
        
        # Redimensionar la imagen a 48x48 píxeles
        resized_img = cv2.resize(gray_img, (48, 48))  
        
        # Convertir la imagen de escala de grises a RGB
        rgb_img = cv2.cvtColor(resized_img, cv2.COLOR_GRAY2RGB)
        
        # Normalizar la imagen (dividir por 255 para que los valores estén entre 0 y 1)
        normalized_img = rgb_img / 255.0

        # Devolver la imagen normalizada (48x48x3) y la imagen redimensionada
        return np.expand_dims(normalized_img, axis=0), resized_img  
    else:
        return None, None  # Si no se detecta una cara, devolver None

while True:
    ret, frame = cap.read()  # Capturar un frame de la cámara
    if not ret:
        print("Error: No se pudo capturar el frame.")
        break

    # Preprocesar la imagen como se hizo durante el entrenamiento
    preprocessed_frame, resized_img = preprocess_image(frame)

    if preprocessed_frame is not None:
        # Predicción de la emoción
        prediction = model.predict(preprocessed_frame)  # Preprocessed_frame ya es RGB y normalizado
        predicted_emotion = emotion_mapper[np.argmax(prediction)]

        # Guardar el preprocessed_frame (debe reescalar de nuevo entre 0-255 y convertir a uint8)
        preprocessed_img_to_save = (preprocessed_frame[0] * 255).astype('uint8')  # Convertir de 0-1 a 0-255
        image_path = os.path.join(output_folder, f'preprocessed_frame_{image_counter}_{predicted_emotion}.jpg')
        cv2.imwrite(image_path, preprocessed_img_to_save)  # Guardar la imagen en formato JPG
        print(f"Imagen preprocesada guardada en: {image_path}")
        image_counter += 1  # Aumentar el contador de imágenes guardadas

        # Mostrar la emoción en el video en tiempo real
        cv2.putText(frame, f'Emocion: {predicted_emotion}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Mostrar el video en tiempo real
    cv2.imshow("Video en Tiempo Real", frame)

    # Presionar 'q' para salir
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar la cámara y cerrar ventanas
cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_0_anger.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_1_neutral.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_2_neutral.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_3_neutral.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_4_neutral.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_5_neutral.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_6_anger.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Imagen preprocesada guardada en: preprocessed_images/preprocessed_frame_7_neutral.jpg
1/1 ━━━━━━━

KeyboardInterrupt: 

In [4]:
cap.release()
cv2.destroyAllWindows()


In [ ]:
# Intentar guardar un frame original para ver si se guarda correctamente

!kaggle kernels output muhammedaymancs/speech-emotion-recognition-ser -p /path/to/dest